<a href="https://colab.research.google.com/github/Yash-Yelave/Global_Gatway_RS/blob/main/NLP_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [3]:
!pip install textblob spacy pandas -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 96.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Imports & File Upload

In [4]:
import pandas as pd
import spacy
from textblob import TextBlob
from datetime import datetime
from dateutil import parser as dateparser
import json
import math
import warnings
import io
from google.colab import files

warnings.filterwarnings("ignore")

# Load NLP Model
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")
print("✓ spaCy loaded\n")

# Prompt for file upload
print("Please upload your 'news_cleaned.csv' from Phase 2...")
uploaded = files.upload()

# Read the uploaded CSV
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f"\n✓ Successfully loaded {len(df)} articles from {filename}.")

# Create a combined text column for NLP processing
df['full_text'] = df['title'].fillna("") + ". " + df['description'].fillna("")

Loading spaCy model...
✓ spaCy loaded

Please upload your 'news_cleaned.csv' from Phase 2...


Saving news_cleaned.csv to news_cleaned.csv

✓ Successfully loaded 250 articles from news_cleaned.csv.


The Core NLP Pipeline

In [5]:
# ==========================================
# 1. KEYWORD EXTRACTION (NER + Noun Chunks)
# ==========================================
def extract_keywords(text):
    if not str(text).strip() or pd.isna(text):
        return []

    doc = nlp(str(text)[:2000]) # Cap at 2000 chars for performance
    keywords = set()

    # Extract Named Entities
    valid_ents = {'PERSON', 'ORG', 'GPE', 'LOC', 'PRODUCT', 'EVENT'}
    for ent in doc.ents:
        if ent.label_ in valid_ents:
            keywords.add(ent.text.lower().strip())

    # Extract Noun Chunks (ignoring isolated pronouns/determiners)
    for chunk in doc.noun_chunks:
        if chunk.root.pos_ != 'PRON' and len(chunk.text) > 2:
            clean_chunk = " ".join([t.text for t in chunk if t.pos_ != 'DET'])
            if clean_chunk:
                keywords.add(clean_chunk.lower().strip())

    return list(keywords)

print("1/4 Extracting keywords using NER and Noun Chunks...")
df['nlp_keywords'] = df['full_text'].apply(extract_keywords)

# ==========================================
# 2. SENTIMENT ANALYSIS (TextBlob)
# ==========================================
def get_textblob_sentiment(text):
    if pd.isna(text):
        return 0.0, "neutral"

    polarity = TextBlob(str(text)).sentiment.polarity
    if polarity > 0.05:
        label = "positive"
    elif polarity < -0.05:
        label = "negative"
    else:
        label = "neutral"
    return round(polarity, 4), label

print("2/4 Computing TextBlob sentiment scores...")
sentiment_results = df['full_text'].apply(get_textblob_sentiment)
df['tb_sentiment_score'] = [res[0] for res in sentiment_results]
df['tb_sentiment_label'] = [res[1] for res in sentiment_results]

# ==========================================
# 3. AUTO-CATEGORIZATION (Keyword Mapping)
# ==========================================
CATEGORY_MAPPING = {
    "technology": ["ai", "artificial intelligence", "software", "apple", "google", "cybersecurity", "tech", "app", "cloud", "crypto"],
    "business": ["economy", "stock", "market", "startup", "ceo", "funding", "revenue", "investor", "wall street", "inflation"],
    "politics": ["government", "election", "president", "senate", "law", "policy", "congress", "diplomacy", "minister"],
    "sports": ["football", "basketball", "fifa", "nba", "championship", "tournament", "coach", "league", "olympics"],
    "health": ["vaccine", "covid", "fda", "hospital", "mental health", "nutrition", "disease", "treatment", "research"]
}

def auto_categorize(keywords):
    scores = {cat: 0 for cat in CATEGORY_MAPPING.keys()}
    for kw in keywords:
        for category, target_words in CATEGORY_MAPPING.items():
            if any(target in kw for target in target_words):
                scores[category] += 1

    best_category = max(scores, key=scores.get)
    return "general" if scores[best_category] == 0 else best_category

print("3/4 Auto-categorizing articles based on keywords...")
df['auto_category'] = df['nlp_keywords'].apply(auto_categorize)

# ==========================================
# 4. FRESHNESS / DECAY SCORE
# ==========================================
today = datetime.utcnow().date()
DECAY_RATE = 0.1

def calculate_freshness(date_val):
    if not date_val or pd.isna(date_val):
        return 0.5
    try:
        pub_date = dateparser.parse(str(date_val)).date()
        days_old = max(0, (today - pub_date).days)
        score = math.exp(-DECAY_RATE * days_old)
        return round(score, 4)
    except:
        return 0.5

print("4/4 Calculating freshness decay scores...")
df['freshness_decay_score'] = df['publish_date'].apply(calculate_freshness)

print("\n✓ NLP Pipeline Processing Complete.")

1/4 Extracting keywords using NER and Noun Chunks...
2/4 Computing TextBlob sentiment scores...
3/4 Auto-categorizing articles based on keywords...
4/4 Calculating freshness decay scores...

✓ NLP Pipeline Processing Complete.


Format & Download File

In [6]:
# Serialize the keyword array to JSON so it saves cleanly in a CSV cell
df['nlp_keywords_json'] = df['nlp_keywords'].apply(json.dumps)

# Clean up temporary processing columns
df = df.drop(columns=['full_text', 'nlp_keywords'])

output_filename = "news_nlp_enriched.csv"
df.to_csv(output_filename, index=False)

print("="*60)
print(f"Saved {len(df)} articles to {output_filename}")
print("New columns added:")
print(" - tb_sentiment_score\n - tb_sentiment_label\n - nlp_keywords_json\n - auto_category\n - freshness_decay_score")
print("="*60)

# Trigger the download in Colab
print("\nDownloading file...")
files.download(output_filename)

Saved 250 articles to news_nlp_enriched.csv
New columns added:
 - tb_sentiment_score
 - tb_sentiment_label
 - nlp_keywords_json
 - auto_category
 - freshness_decay_score



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>